<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/batch_working_capstone_proj_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SEC Edgar Dowloader

In [ ]:
#keep
!pip install sec_edgar_downloader



Required import for file copying and downloads

In [2]:
import os
import shutil
from google.colab import drive

Credentials For Gemini Vertex API for Q&A Pairs Generation

In [5]:
#keep
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = \
    "/content/drive/MyDrive/capstone_qa_data_new/triple-mountain-483601-k3-3a823d61bdb7.json"

Response Data Structure Of Q&A Pair response

In [ ]:
#keep
!pip install google-genai
from pydantic import BaseModel
from typing import List
from google import genai

from pydantic import BaseModel

class QA(BaseModel):

    question:str
    answer:str

class ChunkResult(BaseModel):

    chunk_id:str
    company:str
    year:str
    ticker:str
    section:str
    text:str
    qa_pairs:list[QA]

class BatchResult(BaseModel):

    chunks:list[ChunkResult]

Random return

In [7]:
#keep
import random

def random_flag_percent(random_modulo):
    #TEST SECTION
    #random_int = random.randint(1, 100)
    random_int = random.randint(1, 10)
    #for e.g random_modulo is 5 For 20%
    ret = random_int%(int(random_modulo))
    if(ret == 0):
      return True
    return False

def random_int_range(low, high):
    random_int = random.randint(low, high)
    return random_int

Randomly returns based on percentage of data required

In [8]:

#keep
def create_data_flag(chunk_num, section, total_chunks, num_data_created):
    # Need to take 20% of the chunks at random
    # Keep 100% of top 20% and bottom 20%
    # if there are hundred chunks, we need 20 chunks. 20% of 20 is 4
    # 4 records from top and 4 records from bottom are a must. Remaining
    # 12 records out of 92 at random but if total - num_created is less
    #than the desired number of records, then
    #chunk num 87, num created 8, total 92, desired 12

    #if((total - chunk num) <= (desired - num_created))
         #create the record

    desired_chunks = int(total_chunks*(0.2))
    """if(total_chunks >= 400 and total_chunks < 800):
      desired_chunks = int(total_chunks*(0.1))
    if (total_chunks > 800):
      desired_chunks = int(total_chunks*(0.05))"""
    desired_top_bottom_chunks = int(desired_chunks*(0.2))
    if((chunk_num <= desired_top_bottom_chunks) or
      (chunk_num >= (total_chunks - desired_top_bottom_chunks))):
        return True
    #TEST SECTION
    ret =  random_flag_percent(2)
    #ret =  random_flag_percent(5)
    #if(ret == False):
     # if((total_chunks - chunk_num) <= (desired_chunks - num_data_created)):
        #return True
    return ret





Mount The Drive

In [9]:
#keep
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
#keep
#Copy data to drive

def copy_to_drive(source_dir, destination_dir, file_suffix ):
  # 1. Define source and destination folders
  # Replace 'my_folder' with the exact folder path where your .json files currently are
  #source_dir = '.'
  # Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
  #destination_dir = '/content/drive/MyDrive/capstone_chunks_data_new'

  # Create the destination directory if it doesn't exist
  os.makedirs(destination_dir, exist_ok=True)

  # 2. Find and copy all .json files
  for filename in os.listdir(source_dir):
    if filename.endswith(file_suffix):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

  print("All files copied successfully!")

In [ ]:
!pip install sec-api
!pip install langchain-text-splitters


import requests
import json
import csv
from sec_api import ExtractorApi
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime
from google.colab import files

In [ ]:
#This is the new cell with new code
#keep
#Create jsonl file of all the chunks


total_number_of_records = 0
#TEST SECTION
parsed_cutoff_date = datetime.strptime("20231231", "%Y%m%d").date()
parsed_latest_date = datetime.strptime("20251231", "%Y%m%d").date()
#TEST SECTION
#sections = {"1", "1A", "7", "8"}
sections = {"1", "8"}
#ticker_info = {"ticker":"AAPL", "company_name":"Apple Inc.", "CIK":"0000320193"}
headers = {
    "User-Agent": "triple-mountain-483601-k3@appspot.gserviceaccount.com"
}
sec_des = {"1":"Business", "1A":"Risk Factors", "7":"Managements Discussion and Analysis", "8":"Finacial Statements"}
#cik = ticker_info["CIK"]
#company = ticker_info["company_name"]
#ticker = ticker_info["ticker"]
extractor = ExtractorApi(userdata.get('sec_api_key'))
with open('nasdaq2_cik.csv', mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)

    for row in reader:

        all_ticker_records = []
        ticker_rec_num = 0
        print(row)
        ticker = row[0]
        company  = row[1]
        cik = row[2]
        url = "https://data.sec.gov/submissions/CIK" +  cik +".json"
        data = requests.get(url, headers=headers).content
        #print(data)
        json_object = json.loads(data)
        recent = json_object["filings"]["recent"]
        forms = recent["form"]
        accessions = recent["accessionNumber"]
        dates = recent["filingDate"]
        total_number_of_records = 0
        for form, accession, date in zip(
                  forms,
                  accessions,
                  dates):
          date = date.replace("-","")
          parsed_date = datetime.strptime(date, "%Y%m%d").date()
          if ((form == "10-K") & (parsed_date > parsed_cutoff_date) & (parsed_date <= parsed_latest_date)):
            accession = accession.replace("-","")
            filing_url = "https://www.sec.gov/Archives/edgar/data/" + cik[2:] + "/" + accession + "/" + ticker + "-" + date +".txt"
            for section in sections:
              chunk_file_name = (f"{ticker}-{section}-{date[:4]}-chunks.jsonl")
              file_chunks = open(chunk_file_name, "a", encoding="utf-8")
              #print(filing_url, section)
              text = extractor.get_section(
                      filing_url,
                      section,
                      "text"
                    )
              #metadata = "Reference Ticker-" + ticker + " CompanyName-" + company + " Date-" + date + " Section-" + section
              text_splitter = RecursiveCharacterTextSplitter(
                              separators=[
                                "\n\n",
                                "\n",
                                ". ",
                                " ",
                                ""
                                ],
                                chunk_size=500,
                                chunk_overlap=100
                              )
              chunks = text_splitter.split_text(text)
              total_chunks = len(chunks)
              print("1. Number of chunks in " + ticker + " " + section + " " + date[:4] + " " + str(total_chunks))
              chunk_num = 1
              chunk_record = {}
              for chunk in chunks:
                chunk_record["chunk_id"] = f"{ticker}-{section}-{date[:4]}-{chunk_num}"
                print(f"2. {chunk_record["chunk_id"]}")

                chunk_record["company"] = company

                chunk_record["text"] = chunk

                json.dump(chunk_record, file_chunks)
                file_chunks.write("\n")
                chunk_num = chunk_num + 1
                total_number_of_records = total_number_of_records + 1
                #REMOVE LATER
                if(chunk_num > 11):
                  break
              file_chunks.close()
              #close for loop for sections
              #files.download(f"{ticker}-chunks.json")
        print(f"3. Total chunks in {chunk_file_name}- {total_number_of_records}")
copy_to_drive(".","/content/drive/MyDrive/capstone_chunks_data_new","-chunks.jsonl")




In [12]:
#keep
#Generate QA pairs
import random
import time
from google.api_core.exceptions import ResourceExhausted

def generate_response(prompt, client):
    retries = 0
    while retries < 8:
        try:
            return client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config={
                      "response_mime_type": "application/json",
                      "response_schema": BatchResult.model_json_schema()
                        }
            )

        except ResourceExhausted:
            print("Caught Exception")
            wait = min(
                2 ** retries + random.random(),
                60
            )
            print(f"Sleeping {wait:.1f}s")
            time.sleep(wait)
            retries += 1

    raise RuntimeError("Too many retries")

In [43]:
#keep
#create prompt
def create_batch_prompt(batch):
    prompt = """
              You are creating a financial QA dataset.

              For EACH chunk below:

              1. Generate EXACTLY 2 questions. Use the field company and year to generate questions from each chunk.
              2. Extract the exact answer span.
              3. Return JSON.

              """

    for i, chunk in enumerate(batch):
        tok = chunk['chunk_id'].split("-")
        prompt += f"""

                  Chunk {i}
                  Section: {tok[1]}
                  Year: {tok[2]}
                  Company: {chunk['company']}
                  Ticker: {tok[0]}
                  Id: {chunk['chunk_id']}
                  Text:
                  {chunk['text']}
        """

    return prompt

In [46]:
#Read Chunked Data and shortlist the chunks to be recorded for data
import json
from google.colab import files

records = []
source_dir = '/content/drive/MyDrive/capstone_chunks_data_new'
#filenames = ["NVDA-chunks.jsonl","AAPL-chunks.jsonl","AMZN-chunks.jsonl"]

client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
    )
tickers = ["AMZN"]
#sections = ["8","1","1A", "7"]
  #sections = ["8"]
sections = ["1"]
  #sections = ["1A"]
  #sections = ["7"]
#years = ["2021", "2022", "2023", "2024", "2025"]
years = ["2025"]
  #years = ["2024"]
  #years = ["2023"]
  #years = ["2022"]
  #years = ["2021"]
for ticker in tickers:
  for section in sections:
    for year in years:
      file_name = source_dir+"/"+ticker+"-"+section+"-"+year+"-chunks.jsonl"
      qa_shortlist_file = ticker+"-"+section+"-"+year+"-qachunks.jsonl"
      file_qachunks = open(qa_shortlist_file, "a", encoding="utf-8")
      number_of_chunks = 0
      with open(file_name, "r") as f:
        for line in f:
          print (line)
          number_of_chunks = number_of_chunks + 1
        f.close()
      chunk_num = 1
      num_data_created = 0
      with open(file_name, "r") as f1:
        for line in f1:
          j_obj = json.loads(line)
          key = (f"{j_obj["chunk_id"]}")
          key_info = key.split("-")
          chunk_number = int(key_info[3])
          print(f"Chunk number: {chunk_number}")
          if (create_data_flag(chunk_number, section, number_of_chunks , num_data_created)):
            print(f"Chunk number: {chunk_number} made it")
            print(j_obj)
            json.dump(j_obj, file_qachunks)
            file_qachunks.write("\n")
            num_data_created = num_data_created + 1
        f1.close()
        file_qachunks.close()
      print(f"Number of record shortlisted for Q&A in {file_name} is  {num_data_created} out of {number_of_chunks}")


copy_to_drive(".","/content/drive/MyDrive/capstone_qachunks_data_new","-qachunks.jsonl")





{"chunk_id": "AMZN-1-2025-1", "company": "Amazon.com Inc.", "text": "Item 1. Business ##TABLE_END"}

{"chunk_id": "AMZN-1-2025-2", "company": "Amazon.com Inc.", "text": "This Annual Report on Form 10-K and the documents incorporated herein by reference contain forward-looking statements based on expectations, estimates, and projections as of the date of this filing. Actual results and outcomes may differ materially from those expressed in forward-looking statements. See Item 1A of Part I &#8212; &#8220;Risk Factors.&#8221; As used herein, &#8220;Amazon.com,&#8221; &#8220;we,&#8221; &#8220;our,&#8221; and similar terms include Amazon.com, Inc"}

{"chunk_id": "AMZN-1-2025-3", "company": "Amazon.com Inc.", "text": ". and its subsidiaries, unless the context indicates otherwise."}

{"chunk_id": "AMZN-1-2025-4", "company": "Amazon.com Inc.", "text": "General \n\nWe seek to be Earth&#8217;s most customer-centric company. We are guided by four principles: customer obsession rather than compet

In [47]:
#Read Shortlisted Chunked Data And Generate QA Pairs
import json
from google.colab import files

records_qachunks = []

source_dir = '/content/drive/MyDrive/capstone_qachunks_data_new/'
#filenames = ["NVDA-chunks.jsonl","AAPL-chunks.jsonl","AMZN-chunks.jsonl"]

client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
    )
tickers = ["AMZN"]
#sections = ["8","1","1A", "7"]
#sections = ["8"]
sections = ["1"]
#sections = ["1A"]
#sections = ["7"]
#years = ["2021", "2022", "2023", "2024", "2025"]
years = ["2025"]
#years = ["2024"]
#years = ["2023"]
#years = ["2022"]
#years = ["2021"]

for ticker in tickers:
  for section in sections:
    for year in years:
      write_file = f"{ticker}-{section}-{year}.jsonl"
      file_2 = open(write_file, "a", encoding="utf-8")
      read_file_name =  source_dir + f"{ticker}-{section}-{year}-qachunks.jsonl"
      total = 0
      with open(read_file_name, "r") as rf1:
        for line in rf1:
          total = total + 1
          jobj = json.loads(line)
          records_qachunks.append(jobj)
        rf1.close()
        BATCH_SIZE = 10
        if(total < BATCH_SIZE):
          BATCH_SIZE = total - 1

        for i in range(0, total, BATCH_SIZE):

          batch = records_qachunks[i:i+BATCH_SIZE]

          prompt = create_batch_prompt(batch)
          response = generate_response(prompt, client)
          resp_json = response.model_dump_json()
          resp_dict = json.loads(resp_json)
          chunks = resp_dict['parsed']['chunks']
          for chunk in chunks:
            key = f"{chunk["ticker"]}-{chunk["section"]}-{chunk["year"]}"

            qa_pairs = chunk['qa_pairs']
            #print(qa_pairs)
            for qa_pair in qa_pairs:
              qa_pair['label'] = "1"
              qa_pair['reference'] = chunk["text"]
              qa_pair['chunk_id'] = chunk["chunk_id"]
              qa_pair["metadata"] = f"Reference Ticker-{chunk["ticker"]} CompanyName-{chunk["company"]} Date-{chunk["year"]} Section-{chunk["section"]}"
              #print(qa_pair)
              json.dump(qa_pair, file_2)
              file_2.write("\n")
      file_2.close()
  files.download(f"{write_file}")
  copy_to_drive(".","/content/drive/MyDrive/capstone_qa_data_new",".jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Copied: AMZN-1-2025-qachunks.jsonl
Copied: MSFT-1-2025.jsonl
Copied: AMZN-1-2025.jsonl
All files copied successfully!


In [50]:
#Read all the qa pair files and create a file with all positive records
records = []

source_dir = '/content/drive/MyDrive/capstone_qa_data_new'
#filenames = ["NVDA-chunks.jsonl","AAPL-chunks.jsonl","AMZN-chunks.jsonl"]

client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
    )
tickers = ["MSFT", "AMZN"]
#sections = ["8","1","1A", "7"]
#sections = ["8"]
sections = ["1"]
#sections = ["1A"]
#sections = ["7"]
#years = ["2021", "2022", "2023", "2024", "2025"]
years = ["2025"]
#years = ["2024"]
#years = ["2023"]
#years = ["2022"]
#years = ["2021"]

positive_records = {}
for ticker in tickers:
  for section in sections:
    for year in years:
      read_file_name = f"{ticker}-{section}-{year}.jsonl"
      with open(read_file_name, "r") as rf1:
        for line in rf1:
          j_obj = json.loads(line)
          key = f"{ticker}-{section}-{year}"
          if(key not in positive_records):
                positive_records[key] = []
          positive_records[key].append(j_obj)
      rf1.close()
with open("all_positive_records.json", "a", encoding="utf-8") as apr:
    json.dump(positive_records, apr, indent=4)
apr.close()
files.download("all_positive_records.json")
copy_to_drive(".","/content/drive/MyDrive/capstone_qa_data_new",".json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Copied: all_positive_records.json
All files copied successfully!


In [ ]:
#analyze the Q&A pairs to see what is the sample breakdown of the data
import json
master_rec = {}
grand_total_number_of_records = 0
#tickers = ["NVDA","QCOM","AMAT","MU"]
tickers = ["MSFT","AMZN"]
years = ["2025"]
sections = ["1"]
#tickers = ["AMAT"]
for ticker in tickers:
  rec_count = {}
  records = []
  total_recs = 0
  filename = f"{ticker}.json"
  line_num = 0
  with open(filename, "r") as file1:
    print(f"processing {filename}")
    for line in file1:
          line_num = line_num + 1
          #print (f"{line_num}")
          j_obj = json.loads(line)
          records.append(json.loads(line))

    for record in records:
      key = record['metadata']
      #print (f"{key}")
      display_key = key.split(" ")
      tick = (display_key[1].split("-"))[1]
      section = ((display_key[len(display_key)-1]).split("-"))[1]
      date = ((display_key[len(display_key)-2]).split("-"))[1]
      d_key = f"{tick}-{section}-{date}"
      #print (f"{d_key}")
      if d_key not in master_rec:
        master_rec[d_key] = []
      master_rec[d_key].append(record)
      if d_key not in rec_count:
        rec_count[d_key] = 0
      else:
        rec_count[d_key] = rec_count[d_key] + 1
        total_recs = total_recs + 1
        grand_total_number_of_records = grand_total_number_of_records + 1
    print (f"Total number of records for {ticker}: {total_recs}")
    for k in rec_count.keys():
      print(f"Number of records for {k}: {rec_count[k]}")
print(f"Total number of recs: {grand_total_number_of_records}")




In [ ]:
#Read the data from files

#companies = ["AAPL","MSFT","AMZN","NVDA","META","GOOGL","TSLA","AVGO","COST",
 #            "NFLX","AMD","AMAT","ASML","CSCO","QCOM","INTC","INTU","CMCSA",
  #           "TMUS","TXN","ADBE","PANW","AMGN","SBUX","ISRG","MDLZ","GILD",
   #          "BKNG","REGN","VRTX","ADP","MELI","ADI","KLAC","CTAS","SNPS",
    #         "CDNS","MAR","ORLY","NXPI","CRWD","WDAY","CTSH","ROST","LRCX",
     #        "FAST","PAYX","MCHP","AEP"]
import json

negative_records = {}
negative_records["Negative_Records"] = []
# Open the file in read mode ('r')
#with open("output5.json", "r") as file:
    #data = json.load(file)
data = master_rec
#DELETE LATER
for key in master_rec.keys():
  print(f"Key: {key}")
  print(f"Num of record with this key: {len(master_rec[key])}")
#companies = ["AAPL","MSFT","AMZN","NVDA","META"]
companies = ["AAPL","MU", "QCOM", "AMAT", "NVDA"]
#TEST SECTION
years = ["2025", "2024", "2023", "2022", "2021"]
#years = ["2025"]
sections = ["1", "1A", "7", "8"]
#sections = ["1"]
file_records = {}
#Create 25% each of negative records
num_of_each_neg_rec_type = grand_total_number_of_records*(0.95)
i = 0
while i < num_of_each_neg_rec_type:
  #pick 2 companies at ramdom
  company_index_1 = random_int_range(0, (len(companies) - 1 ))
  company_index_2 = random_int_range(0, (len(companies) - 1 ))

  #pick 2 sections at random
  section_index_1 = random_int_range(0, (len(sections) - 1 ))
  section_index_2 = random_int_range(0, (len(sections) - 1 ))
  #pick 2 years at random
  year_index_1 = random_int_range(0, (len(years) - 1 ))
  year_index_2 = random_int_range(0, (len(years) - 1 ))

  #25%
  if(i <  num_of_each_neg_rec_type):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_1] + "-" + years[year_index_1]

  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*2)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_1]+ "-" + sections[section_index_2] + "-" + years[year_index_1]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*3)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_2]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*4)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_2] + "-" + years[year_index_1]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*5)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_2] + "-" + years[year_index_2]
  if(rec_name_1 not in data or rec_name_2 not in data):
    continue
  records_1 = data[rec_name_1]
  records_2 = data[rec_name_2]

  #pick a random record index from first and second file
  #print(f"Getting range for 1: {rec_name_1}")
  #print(f"Getting range for 2: {rec_name_2}")
  record_index_1 = random_int_range(0, (len(records_1) - 1 ))
  record_index_2 = random_int_range(0, (len(records_2) - 1))



  record_3 = records_1[record_index_1]
  print(records_1[record_index_1])
  record_3['answer'] = records_2[record_index_2]['answer']
  record_3['metadata'] = records_2[record_index_2]['metadata']
  record_3['label'] = "0"
  print(record_3)
  negative_records["Negative_Records"].append(record_3)
  i = i+1



#TODO: write negative record to a file
with open("negative_records.json", "w") as file_neg:
    json.dump(negative_records, file_neg, indent=4)
with open("positive_records.json", "w") as file_pos:
    json.dump(data, file_pos, indent=4)
file_pos.close()
file_neg.close()
print(f"Total Negative Records: {i}")


In [5]:
#Check number of records and keys in positive and negative records

total = 0
num_keys = 0
with open("all_records.json", "r") as file1:
      records = json.load(file1)
      print(f"{len(records["All_Records"])}")
      for key in records.keys():
        total = total + len(records[key])
        num_keys = num_keys + 1
print(f"Total: {total}")
print(f"Number of keys: {num_keys}")




17793
Total: 17793
Number of keys: 1


In [35]:
positive_records = {}
positive_records["Positive_Records"] = []
with open("positive_records.json", "r") as file1:
      records = json.load(file1)
      for key in records.keys():
        for record in records[key]:
          positive_records["Positive_Records"].append(record)
      with open("positive_records_singular.json", "w") as file_pos:
        json.dump(positive_records, file_pos, indent=4)
      file_pos.close()

In [3]:
import random
import json

all_records = {}
all_records["All_Records"] = []
with open("positive_records_singular.json", "r") as file1:
  with open("negative_records.json", "r") as file2:
      records_negative = json.load(file2)
      records_positive = json.load(file1)
      rec_pos = records_positive["Positive_Records"]
      rec_neg = records_negative["Negative_Records"]
      merged = rec_pos + rec_neg
      print(f"Pos length: {len(rec_pos)}")
      print(f"Neg length: {len(rec_neg)}")
      print(f"merged length: {len(merged)}")
      # Shuffle the combined list in place
      random.shuffle(merged)
      i = 0
      print("before Last")
      while(i < len(merged)):
        all_records["All_Records"].append(merged[i])
        i = i + 1
      with open("all_records.json", "w") as file_pos:
        print("Last")
        json.dump(all_records, file_pos, indent=4)
      file_pos.close()

Pos length: 9161
Neg length: 8632
merged length: 17793
before Last
Last
